In [1]:
import os
import csv
import datetime as dt
import pandas as pd

# index를 날짜로 하는 데이터프레임을 price, adj_price, divend에 대해 반환
def extract_stock_data(file_path, ticker):
    extract_datas = []

    with open(os.path.join(file_path, ticker + '.csv'), newline='') as csvfile:
        reader = csv.reader(csvfile)

        for row in reader:
            extract_datas.append(row)
        
    price_dates = list(map(dt.date.fromisoformat, extract_datas[0]))
    price_history = list(map(float, extract_datas[1]))
    adj_price_dates = list(map(dt.date.fromisoformat, extract_datas[2]))
    adj_price_history = list(map(float, extract_datas[3]))
    divend_dates = list(map(dt.date.fromisoformat, extract_datas[4]))
    divend_history = list(map(float, extract_datas[5]))

    price_df = pd.DataFrame({'date': price_dates, 'price': price_history})
    adj_price_df = pd.DataFrame({'date': adj_price_dates, 'adj_price': adj_price_history})
    divend_df = pd.DataFrame({'date': divend_dates, 'divend': divend_history})

    price_df.set_index('date', inplace=True)
    adj_price_df.set_index('date', inplace=True)
    divend_df.set_index('date', inplace=True)

    return (price_df, adj_price_df, divend_df)

def make_stock_data(file_path, tickers):
    stock_data = []

    for ticker in tickers:
        price_df, adj_price_df, divend_df = extract_stock_data(file_path, ticker)
        stock_df = pd.concat([price_df, adj_price_df, divend_df], axis=1, sort=True)
        stock_df.columns = pd.MultiIndex.from_product([[ticker], stock_df.columns])
        stock_data.append(stock_df)

    stock_data = pd.concat(stock_data, axis=1, sort=True)

    return stock_data


In [2]:
from portfolio import Portfolio

# stock 불러와서 데이터프레임화 하기
file_path = './etf'
tickers = ['QQQ', 'DGRW', 'SCHD', 'SPY', 'SCHG', 'SPYG']
stock_data = make_stock_data(file_path, tickers)
pd.options.display.float_format = '{:.2f}'.format

# test_stock = Portfolio(make_stock_data(file_path, ['QQQ', 'SCHG']), 10000)
test_stock = Portfolio(stock_data, 10000)
portfolio = Portfolio(stock_data, 10000, {"QQQ":56, "SCHD":24, "DGRW":20})

In [3]:
def calc_target_ratio(base_ratio:tuple, etc_ratio:tuple) -> tuple:
    base_sum = sum(base_ratio)
    qqq = base_ratio[0] / base_sum * 100
    etc_sum = sum(etc_ratio)
    etc1 = (etc_ratio[0] / etc_sum) * (base_ratio[1] / base_sum) * 100
    etc2 = (etc_ratio[1] / etc_sum) * (base_ratio[1] / base_sum) * 100

    return (round(qqq, 1), round(etc1, 1), round(etc2, 1))

In [4]:
p = Portfolio(stock_data, 10000, {'QQQ':50, 'SCHD':30, 'DGRW':20})
h, s = p.backtest(duration=5, rebalancing_cycle=5, cash_flow=100, cash_flow_growth=10)

In [8]:
h.to_csv('./abc.csv')

In [ ]:
from joblib import Parallel, delayed
from portfolio_test import portfolio_backtest_by_duration
from portfolio_test import make_benchmark_data

stats = pd.DataFrame(columns=['cagr', 'stdev', 'mdd', 'beta', 'alpha'])
example = Portfolio(stock_data, 10000, {'QQQ':50, 'SCHD':30, 'DGRW':20 })
benchmark = Portfolio(stock_data, 10000, {'QQQ':100})
duration = 5
cash_flow = 700

benchmark_data = make_benchmark_data(benchmark, example.start_date, duration, cash_flow)

param_list = []
for qqq_weight in range(0, 100, 5):
    for schd_weight in range(0, 10):
        ratio_qqq_etc = (qqq_weight, 100 - qqq_weight)
        ratio_schd_dgrw = (schd_weight, 10 - schd_weight)
        ratio_qqq_schd_dgrw = calc_target_ratio(ratio_qqq_etc, ratio_schd_dgrw)
        
        target_ratio = {
            'QQQ': ratio_qqq_schd_dgrw[0],
            'SCHD': ratio_qqq_schd_dgrw[1],
            'DGRW': ratio_qqq_schd_dgrw[2]
        }
        print(target_ratio)
        p = Portfolio(stock_data, 10000, target_ratio)
        param_list.append((p, benchmark_data, None, duration, cash_flow))
        break
    break

results = []
for params in param_list:
    results.append(portfolio_backtest_by_duration(*params))
# with Parallel(n_jobs=-1) as parallel:
#         results = parallel(delayed(portfolio_backtest_by_duration)(*params) for params in param_list)
        
for stat in results:
    stats = pd.concat([stats, stat])

stats

{'QQQ': 0.0, 'SCHD': 0.0, 'DGRW': 100.0}
date
2013-05-22       10000
2013-05-23     9976.91
2013-05-24     9971.48
2013-05-28    10036.67
2013-05-29     9989.13
                ...   
2018-05-16   316823.84
2018-05-17   316417.25
2018-05-18   314818.52
2018-05-21   316545.88
2018-05-22   316141.60
Name: (total, value), Length: 1260, dtype: object
               total    cash           QQQ                      SCHD         \
               value   value weight  price number value weight price number   
date                                                                          
2013-05-22     10000       0      0  73.62   0.00  0.00   0.00 11.28   0.00   
2013-05-23   9967.87    0.00   0.00  73.45   0.00  0.00   0.00 11.25   0.00   
2013-05-24   9967.87    0.00   0.00  73.41   0.00  0.00   0.00 11.27   0.00   
2013-05-28  10052.21    0.00   0.00  73.89   0.00  0.00   0.00 11.34   0.00   
2013-05-29   9975.90    0.00   0.00  73.54   0.00  0.00   0.00 11.20   0.00   
...              ..

,cagr,stdev,mdd,beta,alpha,ratio
0,93.67,16.00,10.00,0.81,9.52,00.0:00.0:100.0
1,91.01,16.39,14.36,0.78,11.93,00.0:00.0:100.0
2,89.12,21.02,29.93,0.83,5.98,00.0:00.0:100.0
3,97.58,20.67,30.08,0.78,10.78,00.0:00.0:100.0
4,91.98,20.94,29.13,0.72,16.58,00.0:00.0:100.0
5,91.85,22.29,27.71,0.71,17.43,00.0:00.0:100.0
6,95.21,21.14,23.91,0.70,17.93,00.0:00.0:100.0
7,92.42,17.91,12.84,0.61,23.54,00.0:00.0:100.0
8,94.82,17.67,12.97,0.64,20.44,00.0:00.0:100.0


In [35]:
stats.to_csv('./stats_qqq_schd_dgrw.csv')

In [61]:
stats = pd.read_csv('./stats_qqq_schd_dgrw.csv', index_col=0)
stats

In [96]:
weight_map = {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0,
              5: 1.1, 6: 1.1,
              7: 1.2, 8: 1.2}

stats['weight'] = stats.index.map(lambda x: weight_map[x % 9])
stats

def weighted_alpha_mean(group):
    d = group['alpha']
    w = group['weight']
    return (d * w).sum() / w.sum()

weighted_alpha_stat = stats.groupby('ratio').apply(weighted_alpha_mean)
weighted_alpha_stat = weighted_alpha_stat.sort_values(ascending=False)
mdd_stat = stats.groupby('ratio')['mdd'].quantile(0.25)

pd.set_option('display.max_rows', None)
mdd_stat

ratio
0.0:0.0:100.0    16.93
0.0:10.0:90.0    16.94
0.0:20.0:80.0    16.94
0.0:30.0:70.0    16.97
0.0:40.0:60.0    16.95
0.0:50.0:50.0    16.68
0.0:60.0:40.0    16.66
0.0:70.0:30.0    16.63
0.0:80.0:20.0    16.50
0.0:90.0:10.0    16.38
05.0:0.0:95.0    17.77
05.0:19.0:76.0   17.79
05.0:28.5:66.5   17.80
05.0:38.0:57.0   17.75
05.0:47.5:47.5   17.78
05.0:57.0:38.0   17.50
05.0:66.5:28.5   17.24
05.0:76.0:19.0   17.19
05.0:85.5:9.5    17.11
05.0:9.5:85.5    17.77
10.0:0.0:90.0    18.29
10.0:18.0:72.0   18.59
10.0:27.0:63.0   18.55
10.0:36.0:54.0   18.37
10.0:45.0:45.0   18.31
10.0:54.0:36.0   18.12
10.0:63.0:27.0   18.01
10.0:72.0:18.0   17.90
10.0:81.0:9.0    17.93
10.0:9.0:81.0    18.23
15.0:0.0:85.0    19.22
15.0:17.0:68.0   19.41
15.0:25.5:59.5   19.29
15.0:34.0:51.0   19.21
15.0:42.5:42.5   19.06
15.0:51.0:34.0   18.93
15.0:59.5:25.5   18.81
15.0:68.0:17.0   19.43
15.0:76.5:8.5    18.82
15.0:8.5:76.5    19.41
20.0:0.0:80.0    20.21
20.0:16.0:64.0   20.20
20.0:24.0:56.0   20.12
20.0:

In [101]:
stat_summary = stats.groupby(['ratio']).mean()
stat_summary['alpha'] = weighted_alpha_stat
stat_summary['mdd_median'] = mdd_stat
stat_summary['mdd_diff'] = stat_summary['mdd'] - stat_summary['mdd_median']
stat_summary = stat_summary.sort_values(by='alpha', ascending=False)
stat_summary = stat_summary[stat_summary['alpha'] > 0]
stat_summary.sort_values(by='mdd_diff', ascending=False, inplace=True)
stat_summary[(stat_summary['cagr'] > 14.5)]

,cagr,stdev,mdd,beta,alpha,weight,mdd_median,mdd_diff
ratio,,,,,,,,
35.0:32.5:32.5,14.52,17.34,25.40,0.75,0.13,1.07,22.30,3.10
35.0:39.0:26.0,14.52,17.32,25.43,0.74,0.19,1.07,22.42,3.01
35.0:19.5:45.5,14.59,17.47,25.53,0.76,0.01,1.07,22.63,2.90
35.0:26.0:39.0,14.57,17.40,25.45,0.75,0.08,1.07,22.55,2.90
40.0:54.0:6.0,14.71,17.52,25.58,0.75,0.26,1.07,22.69,2.89
40.0:48.0:12.0,14.78,17.53,25.56,0.75,0.25,1.07,22.78,2.78
40.0:42.0:18.0,14.82,17.56,25.59,0.76,0.21,1.07,23.12,2.48
40.0:36.0:24.0,14.83,17.58,25.59,0.76,0.16,1.07,23.23,2.36
40.0:30.0:30.0,14.86,17.61,25.60,0.77,0.11,1.07,23.30,2.31


In [41]:
pd.set_option('display.max_rows', None)

stat_summary_sorted = stat_summary.sort_values(by=['alpha'], ascending=False)
stat_summary_sorted = stat_summary_sorted[stat_summary_sorted['alpha'] > 0]
stat_sorted = stats.sort_values(by=['ratio', 'alpha'], ascending=[True, False])

In [ ]:
import seaborn as sns

sns.catplot(data=stats, x='ratio', y='cagr', kind='box')
sns.catplot(data=stats, x='ratio', y='stdev', kind='box')
sns.catplot(data=stats, x='ratio', y='mdd', kind='box')

In [ ]:
sns.relplot(data=test, x='stdev', y='cagr', hue='ratio')

In [ ]:
sns.relplot(data=stats, x='stdev', y='cagr', hue='ratio')